# Google Trend Archive Tutorial

This notebook shows how to load the GoogleTrendArchive data, explore it, expand trends with Wikipedia/Wikidata labels, and build a simple digital geography network.

## 1. Environment and package setup

Install or import the packages needed for data loading, text matching, plotting, and network analysis.

In [ ]:
# ! pip install pycountry
# ! pip install pycountry_convert
# ! pip install flashtext
# ! pip install pygraphviz
# ! pip install python-louvain
# ! pip install pandas
# ! pip install seaborn
# ! pip install plotly
# ! pip install requests
# ! pip install tqdm
# ! pip install fsspec
# ! pip install "fsspec>=2024.2.0" huggingface_hub
# ! pip install pyarrow
# ! pip install -U pandas pyarrow
# ! pip install community
# ! pip install -U nbformat ipython
# ! pip install swifter
# ! pip install community
# ! pip install python-louvain
# ! pip install adjustText

Load the main libraries used throughout the notebook.

In [ ]:
import pandas as pd
import pycountry
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pycountry_convert
from itertools import chain
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import date2num
import plotly.express as px
import re 
import requests
from tqdm import tqdm
import pyarrow as pa
from flashtext import KeywordProcessor
import swifter


In [ ]:
import plotly.io as pio
pio.renderers.default = "browser"

### Location metadata

Load the location lookup table and merge it later with the main trend dataset.

In [ ]:
locations = pd.read_csv("hf://datasets/aurman/GoogleTrendArchive/Trends_LocationList.csv")
locations.rename(columns={'location': 'location_name', 'tag':'location'}, inplace=True)
locations.to_csv("./data/Trends_LocationList.csv", index=False)
locations.shape[0]


### Quick location check

A small sanity check for the location table.

In [ ]:
locations[locations.location_name.str.contains('Green')]

### Main dataset source

The raw file path is kept here as a reference for the original preprocessing step.

In [ ]:
df = pd.read_csv("hf://datasets/aurman/GoogleTrendArchive/googletrendarchive_preprocessed_170526.csv")
df.to_parquet('./data/googletrendarchive_preprocessed_170526.parquet', engine='pyarrow', index=False)

## 2. Helper functions

Reusable functions for plotting timelines and for working with the country and topic labels.

### Function definitions

Reusable helpers for plotting timelines and working with country/topic labels.

### Timeline plotting helper

This helper standardizes the time axis and draws the most frequent entities for a given grouping.

In [ ]:
def trends_timeline(_df2, yaxis='location_name', max_entities=20):
    
    _df2["start_time"] = pd.to_datetime(_df2["start_time"])
    _df2["end_time"] = pd.to_datetime(_df2["end_time"])
    _df2.loc[:,'trend_duration'] = _df2.groupby(yaxis)['duration_hours'].transform('sum').copy()

    # Y positions for each location
    locations = _df2[yaxis].dropna().unique().tolist()

    # Use matplotlib's default color cycle for countries
    countries = _df2[yaxis].dropna().astype(str).unique().tolist()
    default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    country_color = {
        country: default_colors[i % len(default_colors)]
        for i, country in enumerate(countries)
    }
    fig, ax = plt.subplots(figsize=(20, 10))

    # small offsets so multiple rows for same location are visible
    offset_step = 0.02

    _order = _df2.groupby(yaxis)['trend_duration'].sum().sort_values(ascending=False)
    _order = _order.head(min(max_entities, len(_order)))
    y_map = {loc: i for i, loc in enumerate(_order.index)}

    for iloc, loc in enumerate(_order.index):
        g = _df2[_df2[yaxis] == loc].copy()

        base_y = y_map[loc]
        g = g.reset_index(drop=True)

        for i, row in g.iterrows():
            y = base_y + (iloc) * offset_step
            ax.hlines(
                y=y,
                xmin=row["start_time"],
                xmax=row["end_time"],
                linewidth=6,
                color=country_color.get(str(row[yaxis]), "C0"),
            )

    # Y-axis labels
    ax.set_yticks(list(y_map.values()))
    ax.set_yticklabels(list(y_map.keys()))

    ax.set_xlabel("Time")
    ax.set_ylabel("Location")
    ax.set_title("Time ranges by location")

    # Legend for countries
    handles = [
        plt.Line2D([0], [0], color=country_color[c], lw=10, label=c)
        for c in countries
    ]
    # ax.legend(handles=handles, title="Country", bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    plt.show()

## 3. Load and inspect the data

Load the parquet file, merge in location names, and keep country-level observations for the analysis.

### Data loading block

The cells below load the main parquet file and prepare the country-level analysis sample.

### Data loading and country flag

Read the parquet file, attach location names, and mark whether the location code is a country.

In [ ]:
google_df = pd.read_parquet('./data/googletrendarchive_preprocessed_170526.parquet', engine='pyarrow')
google_df = google_df.merge(locations, how='left', on='location')
google_df['is_country'] = google_df['location'].apply(lambda x: True if pycountry.countries.get(alpha_2=x) else False)
google_df.location.nunique(), google_df.location_name.nunique()

### Initial data exploration

A few quick checks help verify that the data loaded correctly before moving into the analysis.

**Suggested checks:** number of rows, number of unique locations and trends, date range, missing values, and the most common countries.

In [ ]:
data = google_df = google_df[google_df.is_country].copy()
data['continent'] = data['location'].apply(lambda x: pycountry_convert.country_alpha2_to_continent_code(x))  
data.trends.nunique()

In [ ]:
# Basic shape and coverage
print("Rows, columns:", data.shape)
print("Unique locations:", data["location_name"].nunique())
print("Unique trends:", data["trends"].nunique())

# Date range
data["start_time"] = pd.to_datetime(data["start_time"], errors="coerce")
data["end_time"] = pd.to_datetime(data["end_time"], errors="coerce")
print("Start time range:", data["start_time"].min(), "to", data["start_time"].max())
print("End time range:", data["end_time"].min(), "to", data["end_time"].max())

# Missing values in key fields
key_cols = ["location", "location_name", "trends", "start_time", "end_time", "duration_hours"]
print("\nMissing values in key columns:")
print(data[key_cols].isna().sum().sort_values(ascending=False))

# Simple descriptive statistics
print("\nDuration summary:")
print(data["duration_hours"].describe(percentiles=[0.25, 0.5, 0.75]).round(2))

# Most frequent countries in the analysis sample
print("\nTop 10 countries by number of observations:")
print(data["location_name"].value_counts().head(10))


### Duration distribution overview

The histogram below gives a quick view of how long trends typically remain active across countries.

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))
sns.histplot(data=data, x='duration_hours', bins=100, stat='percent', color='blue', alpha=0.5, element='step', fill=False, ax=ax)
ax.set_xlabel('Trend Duration (hour)')
ax.set_ylabel('% Trends')  

### Country comparison

This comparison is useful for checking whether the duration patterns differ across major countries.

In [ ]:
target_countries = ['United States', 'Germany', 'France', 'United Kingdom', 'Italy', 'Spain', 'Canada', 'Australia', 'Japan', 'Brazil']
fig, ax = plt.subplots(figsize=(6,3))
ax = sns.histplot(data=data[data['location_name'].isin(target_countries)], hue='location_name', x='duration_hours', bins=100, stat='percent', common_norm=False, element='step', fill=False, cumulative=True)  
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
ax.set_xlabel('Trend Duration (hour)')


### Country duration distribution by continent

The boxen plot helps compare duration distributions across countries while also showing continent-level grouping.

In [ ]:
fig, ax = plt.subplots(figsize=(20, 6))
_orders = data.groupby('location_name')['duration_hours'].agg('median').sort_values(ascending=False).index
sns.boxenplot(data=data, x='location_name', y='duration_hours', order=_orders, ax =ax, hue='continent', dodge=False)
_ = plt.xticks(rotation=90)

## 4. Topic expansion with Wikipedia and Wikidata

The next section expands a topic into multilingual labels collected from Wikipedia/Wikidata. This makes it easier to match trends written in different languages or scripts.

### Keyword expansion block

This section turns a Wikipedia topic into multilingual labels and uses them as matching keywords.

### Get the Wikidata QID for a Wikipedia page

If the keyword contains spaces, use the multi-word title exactly as written so the Wikipedia lookup can resolve the right page.

In [ ]:
def get_qid(page_title):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "titles": page_title,
        "prop": "pageprops",
        "format": "json"
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; my-research-script/1.0; +your_email@example.com)"
    }

    r = requests.get(url, params=params, headers=headers)
    response = r.json()

    pages = response["query"]["pages"]

    for page in pages.values():
        return page.get("pageprops", {}).get("wikibase_item")
    print('No Wiki page found.')
    return None

### Fetch all labels for a Wikidata entity

This function returns labels in many languages, which later become candidate keywords for matching trends.

In [ ]:

def fetch_wikidata_labels(qid: str):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetentities",
        "ids": qid,
        "props": "labels",
        "languages": "all",
        "format": "json",
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; my-research-script/1.0; +your_email@example.com)"
    }

    r = requests.get(url, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    data = r.json()

    labels = data["entities"][qid]["labels"]
    df = pd.DataFrame(
        [{"language": lang, "label": info["value"]} for lang, info in labels.items()]
    ).sort_values("language")

    return df

### Small helper for country codes

Convert ISO-2 country codes to ISO-3 when needed.

In [ ]:
def iso2_to_iso3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None


### Build a keyword list from multilingual labels

When `spaced_only=True`, only labels containing spaces are kept. This is useful for multi-word topics such as **Pope Francis** because it reduces false positives from single words, and it makes the note about `spaced_only` explicit: if the keyword has spaces, set this argument to `True`.

You can also be creative here and experiment by changing the parameters. For example, try different topics, switch `spaced_only` on or off, or chain one keyword filter into another to create an implicit **AND** condition.

In [ ]:
def get_related_trends(topic, data, spaced_only=True):
    ID = get_qid(topic)       
    df = fetch_wikidata_labels(ID)
    df['label_lower'] = df['label'].apply(str.lower)
    if spaced_only:
        topic_keywords = df[df['label_lower'].str.contains(' ')]['label_lower'].to_list()
    else:
        topic_keywords = df['label_lower'].to_list()
    pattern = "|".join(re.escape(k) for k in topic_keywords)
    data_keywords = data[data["trends"].str.contains(pattern, case=False, na=False)]
    return data_keywords

### Plot matching trends on a map

This visualizes where the matched topic appears across countries.

In [ ]:
def plot_map(data_keywords):
    _df = data_keywords.groupby(['location_name','location'])['trend_breakdown'].count().reset_index()
    _df["country_code3"] = _df["location"].apply(iso2_to_iso3)

    fig = px.choropleth(
        _df,
        locations="country_code3",
        locationmode="ISO-3",   # change to "ISO-2" if needed
        color="trend_breakdown",
        color_continuous_scale="Viridis",
        projection="natural earth",
        title="Trend Values by Country"
    )

    fig.update_layout(
        geo=dict(showframe=False, showcoastlines=True),
        coloraxis_colorbar=dict(title="Trends")
    )

    fig.show()

### Example 1: Pope Francis

This example uses a multi-word topic, so `spaced_only=True` helps keep the match tighter. A useful trick is to treat the output of one keyword filter as the input to another filter when you want an **AND**-style restriction. The same idea can be used for topics such as **DeepSeek** when you want to narrow the result with a second keyword.

In [ ]:
topic = "Pope Francis"
data_keywords = get_related_trends(topic=topic, data=data, spaced_only=True)
trends_timeline(data_keywords.copy(), yaxis='location_name', max_entities=25)
plot_map(data_keywords)

### Example 2: Pakistan and India

Here two keyword filters are chained together. The second one narrows the result of the first, so the two keywords act like an **AND** condition. You can swap in other country pairs or topic combinations to explore different cases.

In [ ]:
topic = "Blackout"
data_keywords = get_related_trends(topic=topic, data=data, spaced_only=False)
topic2 = "Spain"
data_keywords = get_related_trends(topic=topic2, data=data_keywords, spaced_only=False)
trends_timeline(data_keywords.copy(), yaxis='location_name', max_entities=25)
plot_map(data_keywords)

## 5. Multilingual country dictionaries

The next step collects many language variants of conflict-related countries. For simplicity, the notebook focuses on the countries listed here, but you can add any other country or region by extending the list.

In [ ]:
country_names = pd.read_csv('./data/country-codes.csv')
country_column_names = ['CLDR display name', 'UNTERM English Short', 'official_name_en', 'ISO3166-1-Alpha-2', 'ISO3166-1-Alpha-3']
country_names = country_names[country_column_names]
country_names.loc[country_names['CLDR display name'].isna(),'CLDR display name'] = country_names['official_name_en']

### Target countries list

The current list is intentionally limited to countries involved in conflicts and geopolitically salient cases. Feel free to add other countries, territories, or regions if your research question needs them.

In [ ]:
targeting_countries = [
    'Iran',
    'Israel',
    'Palestine',
    'India',
    'Pakistan',
    'Ukraine',
    'Russia',
    'Thailand',
    'Cambodia',
    'Yemen',
    'Syria',
    'Venezuela',
    'Gaza'
]

### Collect multilingual labels for each country

This loop queries Wikipedia/Wikidata for each country and stores the labels in one table. It is a good place to expand the list if you want to include more countries or languages.

In [ ]:
total_country_keywords = []
for country in tqdm(targeting_countries):
    ID = get_qid(country)    
    if ID == None:
        print(f'No wikipage related to {country} is found.')   
        continue
    df = fetch_wikidata_labels(ID)
    df['country'] = country
    total_country_keywords.append(df)


### Combine labels into a lookup dictionary

The combined table is turned into a multilingual mapping that later supports regex-based matching.

In [ ]:
total_country_keywords_df = pd.concat(total_country_keywords)
total_country_keywords_df.head()
multi_lingual_countries = dict(zip(total_country_keywords_df['label'].str.lower().to_list(), total_country_keywords_df['country'].to_list()))
universal_country_pattern = re.compile(
    "|".join(
        re.escape(k)
        for k in sorted(multi_lingual_countries.keys(), key=len, reverse=True)
    ),
    flags=re.IGNORECASE,
)

### Build a universal country-name pattern

All labels are compiled into one regex pattern so trend strings can be scanned efficiently.

In [ ]:
all_names = list(chain.from_iterable([ country_names[col].tolist() for col in country_column_names]))
all_names = [name for name in all_names if isinstance(name, str)]
all_names = set(all_names)
universal_country_dict = {}
for col in country_column_names:
    universal_country_dict.update(dict(zip(country_names[col], country_names['CLDR display name'])))
universal_country_dict.update(multi_lingual_countries)


## 6. Regex matching and country extraction

This section uses regex and keyword matching to detect country mentions in trend strings. Matching a large dataset can take a while, especially when swifter applies the regex over many rows.

### Extract countries from trend text

The regex-based match is applied row by row. Depending on dataset size, this step may take a while to finish.

In [ ]:
def extract_matches(text):
    if pd.isna(text):
        return []
    return universal_country_pattern.findall(str(text))

data["trend_countries_matches"] = data["trends"].apply(extract_matches)

### Normalize extracted matches

The raw matches are mapped back to the country names used in the analysis.

In [ ]:
data['trend_countries'] = data['trend_countries_matches'].apply(lambda x: sorted({multi_lingual_countries.get(c.lower(),c) for c in x}))
data['ntrend_countries'] = data['trend_countries'].apply(lambda x: len(x))

### Most repeated country combinations

This block summarizes the most common country combinations found in the matched trends.

In [ ]:
_df = data.copy()
_df['countries'] = _df['trend_countries'].apply(lambda x: ','.join(x))
_most_repeated_countries = _df[_df['ntrend_countries']>1].groupby('countries')[['ntrend_countries','duration_hours']].agg({
    'ntrend_countries':  'count',
    'duration_hours': 'sum'
}).sort_values(by='ntrend_countries', ascending=False).head(10)
_most_repeated_countries

### Duration histogram for repeated combinations

A small follow-up visualization for the most repeated country combinations.

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.histplot(_df[_df['countries'].isin(_most_repeated_countries.index)], x='duration_hours', hue='countries', \
             element='step', fill=False, cumulative=True, common_norm=False, bins=40)

### Sanity check on extracted country names

Inspect the distinct country labels found in the dataset.

In [ ]:
total_country_keywords_df['country'].unique()

### Example 3: India and Pakistan

This example looks for trends where both countries appear together. The same chaining idea can be used for other pairs of countries or topics.

In [ ]:
_df = data[data['trend_countries'].apply(lambda x: 'India' in x and 'Pakistan' in x )].copy()
trends_timeline(_df, yaxis='location_name')
plot_map(_df)

### Example 4: Iran and Israel

Another paired example showing how to filter for two keywords at once.

In [ ]:
_df = data[data['trend_countries'].apply(lambda x: "Iran" in x and 'Israel' in x)].copy()
trends_timeline(_df, yaxis='location_name')
plot_map(_df)

In [ ]:
_df = data[data['trend_countries'].apply(lambda x: "Cambodia" in x and 'Thailand' in x)].copy()
trends_timeline(_df, yaxis='location_name')
plot_map(_df)

## 7. Digital geography network

The final section builds a network of countries connected by shared trends.

## 7. Digital geography network

This section builds a network of countries linked by shared trends.

### Network setup

Import the graph tools needed to build and analyze country-to-country connections.

In [ ]:
import networkx as nx
from itertools import combinations

### Build the graph of shared trends

Each edge represents a pair of countries; the edge weight is the number of trends they share.

In [ ]:
# -------- All Tuples ---------
# data2 = google_df.dropna(subset=["location_name", "trends"]).copy()
# edge_threshold = 300

# -------- Excludive Tuples ---------
data2 = google_df.dropna(subset=["location_name", "trends"]).copy()
_df = data2.groupby('trends')[['location']].agg('count').reset_index()
_df = _df[_df['location']==2].copy()
data2 = data2.merge(_df[['trends']], on='trends')
edge_threshold = 1


# 1) Build a set of unique trends for each location
location_trends = (
    data2.groupby("location_name")["trends"]
    .apply(lambda s: set(s.astype(str).str.strip()))
    .to_dict()
)

# 2) Create graph
G = nx.Graph()

# Add nodes
for location in location_trends:
    G.add_node(location)

# Add weighted edges based on intersection size
for loc1, loc2 in combinations(location_trends.keys(), 2):
    mutual_trends = location_trends[loc1].intersection(location_trends[loc2])
    weight = len(mutual_trends)
    G.add_edge(loc1, loc2, weight=weight)

### Inspect the strongest connections

Printing the heaviest edges is a quick way to see which country pairs share the most trends.

In [ ]:
top10 = sorted(
    G.edges(data=True),
    key=lambda x: x[2]["weight"],
    reverse=True
)[:15]

for u, v, d in top10:
    print(f"{u:15} -- \t{v:25} # mutual trends = {d['weight']}")

### Example intersection check

A small manual check for a specific pair of countries.

In [ ]:
import random
loc1, loc2 = 'Canada', "United Kingdom"
list(location_trends[loc1].intersection(location_trends[loc2]))[:20]

### Community detection

Use Louvain clustering to identify groups of countries that share similar trend patterns.

In [ ]:
from community import community_louvain

partition = community_louvain.best_partition(
    G,
    weight="weight",
    random_state=42,
)

# node -> community id
colors = [partition[n] for n in G.nodes()]


### Network visualization

Draw the graph with a more readable layout and lighter edges so dense networks stay interpretable.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from adjustText import adjust_text
import numpy as np

fig, ax = plt.subplots(figsize=(14, 14))

# More spread-out layout
pos = nx.spring_layout(G, seed=42, k=2, iterations=700)

strong_edges = [
    (u, v) for u, v, d in G.edges(data=True)
    if d.get("weight", 0) > edge_threshold
]

weights = np.array([G[u][v]["weight"] for u, v in strong_edges], dtype=float)

# Make widths much narrower and less extreme
# log scaling helps when weights vary a lot
edge_widths = 0.3 + 1.2 * (np.log1p(weights - edge_threshold) / np.log1p(weights.max() - edge_threshold))

# Use a light gray edge color instead of black
# alpha keeps the graph from becoming a black blob
nx.draw_networkx_edges(
    G,
    pos,
    edgelist=strong_edges,
    width=edge_widths,
    edge_color="#666666",
    alpha=0.25,
    ax=ax,
)

# Draw nodes
nx.draw_networkx_nodes(
    G,
    pos,
    node_color=colors,
    node_size=180,
    edgecolors="white",
    linewidths=0.6,
    ax=ax,
)

# Labels with white background
texts = []
for node, (x, y) in pos.items():
    texts.append(
        ax.text(
            x, y, str(node),
            fontsize=12,
            bbox=dict(
                facecolor="white",
                edgecolor="none",
                alpha=0.85,
                pad=0.15,
            ),
        )
    )

# Avoid label overlap
adjust_text(
    texts,
    ax=ax,
    arrowprops=dict(
        arrowstyle="-",
        color="gray",
        lw=0.4,
        alpha=0.4,
    ),
)

ax.set_facecolor("white")
fig.set_facecolor("white")
ax.axis("off")

plt.tight_layout()
plt.show()

## 8. Next improvement

A useful extension is to count only trends that are temporally close to each other, so mutual trends are not just shared in name but also in temporal overlap.